In [0]:
from delta.tables import DeltaTable

In [0]:
DeltaTable.createIfNotExists(spark) \
  .tableName("databricks_streaming.stream.sourcetable") \
  .addColumn("color", "STRING")\
  .execute()

In [0]:
DeltaTable.createIfNotExists(spark) \
  .tableName("databricks_streaming.stream.sinktable") \
  .addColumn("color", "STRING")\
  .execute()

In [0]:
%sql
INSERT INTO databricks_streaming.stream.sourcetable VALUES 
('maroon')

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO databricks_streaming.stream.sourcetable VALUES
    ('red'),
    ('green'),
    ('blue'),
    ('yellow'),
    ('orange'),
    ('orange')

num_affected_rows,num_inserted_rows
6,6


In [0]:
%sql
INSERT INTO databricks_streaming.stream.sourcetable VALUES
    ('red'),
    ('green'),
    ('blue')

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM databricks_streaming.stream.sourcetable

color
red
green
blue
yellow
orange
orange
red
green
blue


In [0]:
df = spark.readStream.table('databricks_streaming.stream.sourcetable')

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.groupBy("color").agg(count("*").alias("count"))

In [0]:
# # check result of above
# (df.writeStream
#   .format("memory")
#   .queryName("color_counts")
#   .outputMode("complete")
#   .option("checkpointLocation", "/Volumes/databricks_streaming/stream/streaming/_checkpoints/color_counts")
#   .trigger(once=True)
#   .start()
# )

In [0]:
# display(spark.sql("SELECT * FROM color_counts"))

color,count
orange,2
green,1
yellow,1
red,1
blue,1


In [0]:
df.writeStream.format("delta")\
            .outputMode('complete')\
            .trigger(once=True)\
            .option("checkpointLocation","/Volumes/databricks_streaming/stream/streaming/output/check")\
            .option("path","/Volumes/databricks_streaming/stream/streaming/output/Data")\
            .start()

In [0]:
%sql
SELECT * FROM delta.`/Volumes/databricks_streaming/stream/streaming/output/Data`

color,count
yellow,1
orange,2
maroon,1
green,2
blue,2
red,2
